# Chapter 28: Agentic UI — The GAS Patient Dashboard

> *"An agent without a UI is a tool. An agent with a great UI is a product. The interface is where AI meets human trust."*

**Book:** Agentic AI — From Foundations to Production  
**Part 5:** Building Production Agentic Systems  
**Chapter:** 28

## 🎯 Chapter Goal

By the end of this chapter you will be able to:

1. **Design** agentic UIs that keep humans in the loop without creating friction
2. **Build** a Streamlit dashboard for the Guardian Angel System
3. **Implement** human approval gates for high-stakes agent actions
4. **Create** real-time agent activity feeds that build user trust
5. **Understand** the Streamlit vs Gradio tradeoff for agentic applications
6. **Apply** the HITL (Human-in-the-Loop) design pattern in a medical context

## 📖 Theory Recap

### Why Agentic UIs Are Different

Traditional UIs are **reactive**: user clicks → system responds. Agentic UIs are **proactive**: the agent acts continuously, and the UI must:

1. **Show agent reasoning** — not just outputs, but *why* the agent made a decision
2. **Enable approval gates** — let humans approve/reject high-stakes actions
3. **Provide override controls** — humans must always be able to override the agent
4. **Build trust through transparency** — show what the agent is doing in real time

### The 4 Agentic UI Patterns

| Pattern | Description | GAS Example |
|---------|-------------|-------------|
| **Activity Feed** | Stream of agent actions | "Monitor checked glucose at 3:42 AM" |
| **Approval Gate** | Human must approve before action | "Approve insulin correction?" |
| **Override Panel** | Human can override agent decision | "Override: patient ate, no correction needed" |
| **Confidence Display** | Show agent certainty | "87% confident: hypoglycemia risk" |

### Streamlit vs Gradio for Agentic Apps

| Dimension | Streamlit | Gradio |
|-----------|-----------|--------|
| **Best for** | Dashboards, complex layouts | Quick demos, model interfaces |
| **State management** | `st.session_state` | `gr.State` |
| **Real-time updates** | `st.empty()` + rerun | Streaming outputs |
| **Custom components** | Full HTML/CSS/JS | Limited |
| **Learning curve** | Medium | Low |
| **Production readiness** | High | Medium |
| **GAS recommendation** | ✅ Use for patient dashboard | Use for quick prototypes |

### The GAS Dashboard Architecture

```
┌─────────────────────────────────────────────────────────────┐
│  Guardian Angel System — Patient Dashboard                  │
├──────────┬──────────────────────────┬────────┬─────────────┤
│ Sidebar  │  Glucose Chart (60%)     │ Status │ Agent Feed   │
│          │                          │  (20%) │    (20%)     │
│ Patient  │  [Plotly chart with      │        │              │
│ Selector │   target range shaded]   │ 🔴 LOW │ 03:42 Mon..  │
│          │                          │ 45 ↓   │ 03:41 Adv..  │
│ Profile  │                          │ ALERT  │ 03:40 Saf..  │
├──────────┴──────────────────────────┴────────┴─────────────┤
│  Recommendation: Consume 15g fast-acting carbs immediately  │
│  [✅ APPROVE]  [❌ REJECT]                                  │
└─────────────────────────────────────────────────────────────┘
```

## 🔨 Build It

### Setup

In [ ]:
import os
import sys
import json
import random
from datetime import datetime, timedelta, timezone
from pathlib import Path

# TEST_MODE: run without any external API keys
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"

print(f"TEST_MODE: {TEST_MODE}")
print("Agentic UI notebook ready.")

### Dashboard Layout (ASCII Mockup)

Before writing any code, let's design the layout:

```
╔══════════════════════════════════════════════════════════════════════╗
║  🏥 Guardian Angel System                          [🔄 Refresh]     ║
╠══════════╦═══════════════════════════════╦═════════╦════════════════╣
║          ║                               ║         ║                ║
║ SIDEBAR  ║   GLUCOSE CHART (60%)         ║ STATUS  ║  AGENT FEED   ║
║          ║                               ║  (20%)  ║    (20%)      ║
║ Patient: ║  300 ┤                        ║         ║               ║
║ ○ Alice  ║  250 ┤         ╭─╮            ║  🔴 LOW ║ 03:42 Monitor ║
║ ○ Bob    ║  200 ┤        ╭╯ ╰╮           ║         ║ checked CGM   ║
║          ║  180 ┤────────╯   ╰──── ─ ─   ║  45     ║               ║
║ Profile: ║  150 ┤  ╭──╮                  ║  mg/dL  ║ 03:41 Advisor ║
║ T1D      ║  100 ┤──╯  ╰──╮              ║         ║ recommended   ║
║ Age: 34  ║   70 ┤─────────╰──────── ─ ─  ║  ↓ Fall ║ 15g carbs     ║
║ ISF: 50  ║   40 ┤                        ║         ║               ║
║          ║      └──────────────────────  ║  ALERT  ║ 03:40 Safety  ║
║          ║      00:00  06:00  12:00  18:00║         ║ approved      ║
╠══════════╩═══════════════════════════════╩═════════╩════════════════╣
║  💊 Recommendation: Consume 15g fast-acting carbs immediately       ║
║  ┌─────────────────────┐  ┌─────────────────────┐                  ║
║  │  ✅ APPROVE         │  │  ❌ REJECT           │                  ║
║  └─────────────────────┘  └─────────────────────┘                  ║
╚══════════════════════════════════════════════════════════════════════╝
```

**Key design decisions:**
- **60% width for chart** — glucose data is the primary information
- **Color-coded status** — red/orange/green at a glance
- **Agent feed** — builds trust by showing what the AI is doing
- **Approval gate** — only shown for high/critical alerts

### The Complete Streamlit App

The full `app.py` is saved at `ch28_agentic_ui/app.py`. Here we walk through each component.

**To run the app:**
```bash
cd /path/to/agentic-ai-book/part5_agentic/ch28_agentic_ui
streamlit run app.py
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# app.py — Complete Streamlit Application
# Guardian Angel System Patient Dashboard
# ─────────────────────────────────────────────────────────────

APP_CODE = '''
"""
app.py — Guardian Angel System Patient Dashboard
================================================
Run with: streamlit run app.py
"""

import sys
import os
from pathlib import Path

# Add shared/ to path so we can import CGMSimulator
repo_root = Path(__file__).parent.parent.parent
sys.path.insert(0, str(repo_root))

import streamlit as st
import plotly.graph_objects as go
from datetime import datetime, timedelta, timezone
import random

# ── Page config ──────────────────────────────────────────────
st.set_page_config(
    page_title="Guardian Angel System",
    page_icon="🏥",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Patient profiles ─────────────────────────────────────────
PATIENTS = {
    "Alice T1D": {
        "id": "alice-001",
        "diabetes_type": 1,
        "age": 34,
        "isf": 50,   # insulin sensitivity factor
        "target_low": 70,
        "target_high": 180,
        "scenario": "hypoglycemia_episode",
        "seed": 42,
    },
    "Bob T2D": {
        "id": "bob-002",
        "diabetes_type": 2,
        "age": 58,
        "isf": 80,
        "target_low": 80,
        "target_high": 200,
        "scenario": "post_meal_spike",
        "seed": 99,
    },
}

# ── Session state initialization ─────────────────────────────
if "approval_history" not in st.session_state:
    st.session_state.approval_history = []
if "agent_feed" not in st.session_state:
    st.session_state.agent_feed = []
if "selected_patient" not in st.session_state:
    st.session_state.selected_patient = "Alice T1D"

# ── Helper functions ─────────────────────────────────────────

def classify_glucose(glucose: float) -> tuple:
    """Return (level_name, color, emoji) for a glucose value."""
    if glucose < 54:
        return "CRITICAL LOW", "#FF0000", "🔴"
    elif glucose < 70:
        return "LOW", "#FF6600", "🟠"
    elif glucose <= 180:
        return "IN RANGE", "#00AA00", "🟢"
    elif glucose <= 250:
        return "HIGH", "#FF6600", "🟠"
    else:
        return "CRITICAL HIGH", "#FF0000", "🔴"


def get_trend_arrow(readings: list) -> str:
    """Get trend arrow from last 2 readings."""
    if len(readings) < 2:
        return "→"
    delta = readings[-1] - readings[-2]
    if delta > 5:
        return "↑↑" if delta > 15 else "↑"
    elif delta < -5:
        return "↓↓" if delta < -15 else "↓"
    return "→"


def get_recommendation(glucose: float, trend: str) -> tuple:
    """Get recommendation and alert level for current glucose."""
    if glucose < 54:
        return (
            "CRITICAL: Consume 15g fast-acting carbs IMMEDIATELY. Recheck in 15 min.",
            "critical"
        )
    elif glucose < 70:
        return (
            "WARNING: Glucose below target. Consume 10-15g carbs. Monitor closely.",
            "high"
        )
    elif glucose > 250:
        return (
            "CRITICAL: Significant hyperglycemia. Check ketones. Consider correction dose.",
            "critical"
        )
    elif glucose > 180:
        return (
            "CAUTION: Glucose above target. Check for missed bolus. Consider correction.",
            "high"
        )
    else:
        return (
            "OK: Glucose within target range. Continue current management.",
            "normal"
        )


def load_glucose_data(patient: dict) -> tuple:
    """Load 24h of glucose data for a patient."""
    try:
        from shared.patient_data import CGMSimulator
        sim = CGMSimulator(
            patient_id=patient["id"],
            diabetes_type=patient["diabetes_type"],
            seed=patient["seed"],
        )
        readings = sim.generate_scenario(patient["scenario"])
        times = [r.timestamp for r in readings]
        values = [r.value_mgdl for r in readings]
        return times, values
    except ImportError:
        # Fallback: generate synthetic data without CGMSimulator
        import math
        now = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0)
        times = [now + timedelta(minutes=5*i) for i in range(288)]
        rng = random.Random(patient["seed"])
        values = []
        base = 105.0
        for i, t in enumerate(times):
            h = t.hour + t.minute / 60
            # Simulate hypoglycemia at 3am
            if 2.5 <= h <= 4.5:
                base_v = 105 - 60 * math.exp(-((h - 3.5)**2) / 0.5)
            else:
                base_v = 105 + 30 * math.sin(h * math.pi / 12)
            values.append(max(40, base_v + rng.gauss(0, 5)))
        return times, values


def add_agent_event(agent: str, message: str):
    """Add an event to the agent activity feed."""
    st.session_state.agent_feed.insert(0, {
        "time": datetime.now().strftime("%H:%M:%S"),
        "agent": agent,
        "message": message,
    })
    # Keep only last 10 events
    st.session_state.agent_feed = st.session_state.agent_feed[:10]


# ── Sidebar ───────────────────────────────────────────────────
with st.sidebar:
    st.title("🏥 GAS Dashboard")
    st.divider()

    # Patient selector
    st.subheader("👤 Patient")
    selected = st.radio(
        "Select patient:",
        options=list(PATIENTS.keys()),
        index=list(PATIENTS.keys()).index(st.session_state.selected_patient),
    )
    st.session_state.selected_patient = selected
    patient = PATIENTS[selected]

    # Patient profile
    st.divider()
    st.subheader("📋 Profile")
    st.metric("Diabetes Type", f"Type {patient['diabetes_type']}")
    st.metric("Age", patient["age"])
    st.metric("ISF", f"{patient['isf']} mg/dL per unit")
    st.metric("Target Range", f"{patient['target_low']}–{patient['target_high']} mg/dL")

    st.divider()
    refresh = st.button("🔄 Refresh", use_container_width=True)

# ── Load data ─────────────────────────────────────────────────
times, values = load_glucose_data(patient)
current_glucose = values[-1]
trend_arrow = get_trend_arrow(values[-5:])
level_name, level_color, level_emoji = classify_glucose(current_glucose)
recommendation, alert_level = get_recommendation(current_glucose, trend_arrow)

# Simulate agent activity on load/refresh
if refresh or not st.session_state.agent_feed:
    add_agent_event("Monitor", f"CGM reading: {current_glucose:.0f} mg/dL ({trend_arrow})")
    add_agent_event("Advisor", f"Generated recommendation: {recommendation[:40]}...")
    add_agent_event("Safety", "Guardrails check: PASSED")
    if alert_level in ("high", "critical"):
        add_agent_event("Caregiver", f"Alert queued: {level_name}")

# ── Header ────────────────────────────────────────────────────
st.title(f"🏥 Guardian Angel System — {selected}")
st.caption(f"Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ── Main layout: 3 columns ────────────────────────────────────
col_chart, col_status, col_feed = st.columns([6, 2, 2])

# ── Column 1: Glucose Chart ───────────────────────────────────
with col_chart:
    st.subheader("📈 Glucose (Last 24h)")

    fig = go.Figure()

    # Target range shading (green)
    fig.add_hrect(
        y0=patient["target_low"], y1=patient["target_high"],
        fillcolor="rgba(0,200,0,0.08)",
        line_width=0,
        annotation_text="Target",
        annotation_position="top left",
    )

    # Danger zones (red)
    fig.add_hrect(
        y0=0, y1=54,
        fillcolor="rgba(255,0,0,0.12)",
        line_width=0,
        annotation_text="Severe Hypo",
        annotation_position="bottom left",
    )
    fig.add_hrect(
        y0=300, y1=500,
        fillcolor="rgba(255,0,0,0.12)",
        line_width=0,
        annotation_text="Severe Hyper",
        annotation_position="top left",
    )

    # Threshold lines
    fig.add_hline(y=patient["target_low"], line_dash="dash", line_color="orange", opacity=0.5)
    fig.add_hline(y=patient["target_high"], line_dash="dash", line_color="orange", opacity=0.5)
    fig.add_hline(y=54, line_dash="dot", line_color="red", opacity=0.5)

    # Glucose trace
    fig.add_trace(go.Scatter(
        x=times, y=values,
        mode="lines",
        name="Glucose",
        line=dict(color="#1f77b4", width=2),
        hovertemplate="%{x|%H:%M}<br>%{y:.0f} mg/dL<extra></extra>",
    ))

    # Current reading marker
    fig.add_trace(go.Scatter(
        x=[times[-1]], y=[current_glucose],
        mode="markers",
        name="Current",
        marker=dict(color=level_color, size=12, symbol="circle"),
        hovertemplate=f"Current: {current_glucose:.0f} mg/dL<extra></extra>",
    ))

    fig.update_layout(
        height=350,
        margin=dict(l=0, r=0, t=10, b=0),
        xaxis_title="Time",
        yaxis_title="Glucose (mg/dL)",
        yaxis=dict(range=[30, max(max(values) + 20, patient["target_high"] + 40)]),
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
        hovermode="x unified",
    )

    st.plotly_chart(fig, use_container_width=True)

# ── Column 2: Status Card ─────────────────────────────────────
with col_status:
    st.subheader("📊 Status")

    # Glucose value with color
    st.markdown(
        f"""
        <div style="
            background-color: {level_color}22;
            border: 2px solid {level_color};
            border-radius: 10px;
            padding: 15px;
            text-align: center;
            margin-bottom: 10px;
        ">
            <div style="font-size: 2.5em; font-weight: bold; color: {level_color};">
                {current_glucose:.0f}
            </div>
            <div style="font-size: 0.9em; color: gray;">mg/dL</div>
            <div style="font-size: 1.5em;">{trend_arrow}</div>
            <div style="font-size: 0.8em; font-weight: bold; color: {level_color};">
                {level_emoji} {level_name}
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    # Stats
    if len(values) > 0:
        in_range = sum(1 for v in values if patient["target_low"] <= v <= patient["target_high"])
        tir = in_range / len(values) * 100
        st.metric("Time in Range", f"{tir:.0f}%")
        st.metric("Min", f"{min(values):.0f} mg/dL")
        st.metric("Max", f"{max(values):.0f} mg/dL")
        st.metric("Mean", f"{sum(values)/len(values):.0f} mg/dL")

# ── Column 3: Agent Activity Feed ────────────────────────────
with col_feed:
    st.subheader("🤖 Agent Feed")

    if not st.session_state.agent_feed:
        st.caption("No activity yet. Click Refresh.")
    else:
        for event in st.session_state.agent_feed:
            agent_emoji = {
                "Monitor": "👁️",
                "Advisor": "💡",
                "Safety": "🛡️",
                "Caregiver": "📱",
            }.get(event["agent"], "🤖")

            st.markdown(
                f"""
                <div style="
                    border-left: 3px solid #1f77b4;
                    padding: 4px 8px;
                    margin-bottom: 6px;
                    font-size: 0.8em;
                ">
                    <span style="color: gray;">{event['time']}</span><br>
                    <strong>{agent_emoji} {event['agent']}</strong><br>
                    {event['message'][:50]}
                </div>
                """,
                unsafe_allow_html=True,
            )

# ── Bottom: Recommendation Panel ─────────────────────────────
st.divider()
st.subheader("💊 Recommendation")

rec_col, btn_col = st.columns([3, 1])

with rec_col:
    alert_colors = {
        "normal": "#00AA00",
        "high": "#FF6600",
        "critical": "#FF0000",
    }
    color = alert_colors.get(alert_level, "#666666")
    st.markdown(
        f"""
        <div style="
            background-color: {color}11;
            border-left: 4px solid {color};
            padding: 12px 16px;
            border-radius: 4px;
            font-size: 1.05em;
        ">
            {recommendation}
        </div>
        """,
        unsafe_allow_html=True,
    )

with btn_col:
    if alert_level in ("high", "critical"):
        st.write("**Human Approval Required**")
        approve_col, reject_col = st.columns(2)
        with approve_col:
            if st.button("✅ APPROVE", type="primary", use_container_width=True):
                st.session_state.approval_history.append({
                    "time": datetime.now().isoformat(),
                    "action": "APPROVED",
                    "glucose": current_glucose,
                    "recommendation": recommendation,
                })
                add_agent_event("Caregiver", "Recommendation APPROVED by human")
                st.success("Approved!")
        with reject_col:
            if st.button("❌ REJECT", use_container_width=True):
                st.session_state.approval_history.append({
                    "time": datetime.now().isoformat(),
                    "action": "REJECTED",
                    "glucose": current_glucose,
                    "recommendation": recommendation,
                })
                add_agent_event("Caregiver", "Recommendation REJECTED by human")
                st.warning("Rejected. Agent will re-evaluate.")
    else:
        st.success("✅ Auto-approved (normal range)")

# ── Approval History ──────────────────────────────────────────
if st.session_state.approval_history:
    with st.expander(f"📋 Approval History ({len(st.session_state.approval_history)} decisions)"):
        for entry in reversed(st.session_state.approval_history[-5:]):
            icon = "✅" if entry["action"] == "APPROVED" else "❌"
            st.write(f"{icon} **{entry['action']}** at {entry['time'][:19]} | Glucose: {entry['glucose']:.0f} mg/dL")
            st.caption(entry["recommendation"][:80])
'''

print("app.py code defined (shown above).")
print(f"Total lines: {len(APP_CODE.strip().splitlines())}")

### Component Deep Dives

Let's examine each key component in isolation.

In [ ]:
# ─────────────────────────────────────────────
# Component 1: Plotly Glucose Chart
# Target range 70-180 shaded green, danger zones red
# ─────────────────────────────────────────────

try:
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("Plotly not installed. Install with: pip install plotly")

import math
from datetime import datetime, timedelta, timezone

# Generate demo glucose data
def generate_demo_glucose(seed=42, scenario="hypoglycemia"):
    """Generate 24h of demo glucose data."""
    rng = random.Random(seed)
    now = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
    times = [now + timedelta(minutes=5*i) for i in range(288)]
    values = []
    for t in times:
        h = t.hour + t.minute / 60
        if scenario == "hypoglycemia" and 2.5 <= h <= 4.5:
            # Nocturnal low
            base_v = 105 - 65 * math.exp(-((h - 3.5)**2) / 0.4)
        elif scenario == "hypoglycemia" and 4.5 < h <= 7:
            # Somogyi rebound
            base_v = 105 + 80 * math.exp(-((h - 5)**2) / 0.5)
        else:
            # Normal day with meals
            meal_effect = (
                50 * math.exp(-((h - 8)**2) / 0.5) +   # breakfast
                60 * math.exp(-((h - 13)**2) / 0.5) +  # lunch
                55 * math.exp(-((h - 19)**2) / 0.5)    # dinner
            )
            base_v = 105 + meal_effect
        values.append(max(40, base_v + rng.gauss(0, 5)))
    return times, values

times, values = generate_demo_glucose()
current_glucose = values[-1]

if PLOTLY_AVAILABLE:
    fig = go.Figure()

    # Target range (green)
    fig.add_hrect(y0=70, y1=180, fillcolor="rgba(0,200,0,0.08)",
                  line_width=0, annotation_text="Target Range",
                  annotation_position="top left")

    # Danger zones (red)
    fig.add_hrect(y0=0, y1=54, fillcolor="rgba(255,0,0,0.12)",
                  line_width=0, annotation_text="Severe Hypo",
                  annotation_position="bottom left")
    fig.add_hrect(y0=300, y1=500, fillcolor="rgba(255,0,0,0.12)",
                  line_width=0, annotation_text="Severe Hyper",
                  annotation_position="top left")

    # Threshold lines
    fig.add_hline(y=70, line_dash="dash", line_color="orange", opacity=0.6)
    fig.add_hline(y=180, line_dash="dash", line_color="orange", opacity=0.6)
    fig.add_hline(y=54, line_dash="dot", line_color="red", opacity=0.6)

    # Glucose trace
    fig.add_trace(go.Scatter(
        x=times, y=values,
        mode="lines",
        name="Glucose",
        line=dict(color="#1f77b4", width=2),
        hovertemplate="%{x|%H:%M}<br>%{y:.0f} mg/dL<extra></extra>",
    ))

    # Current reading
    fig.add_trace(go.Scatter(
        x=[times[-1]], y=[current_glucose],
        mode="markers",
        name="Current",
        marker=dict(color="red", size=12),
    ))

    fig.update_layout(
        title="GAS Patient Dashboard — Glucose Chart (24h)",
        xaxis_title="Time",
        yaxis_title="Glucose (mg/dL)",
        yaxis=dict(range=[30, 320]),
        height=400,
        hovermode="x unified",
    )

    fig.show()
    print(f"Current glucose: {current_glucose:.0f} mg/dL")
else:
    # Fallback: matplotlib
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.axhspan(70, 180, alpha=0.1, color="green", label="Target")
    ax.axhspan(0, 54, alpha=0.12, color="red", label="Severe Hypo")
    ax.plot(range(len(values)), values, "b-", linewidth=1.5)
    ax.set_title("GAS Patient Dashboard — Glucose Chart (24h)")
    ax.set_ylabel("Glucose (mg/dL)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Current glucose: {current_glucose:.0f} mg/dL")

In [ ]:
# ─────────────────────────────────────────────
# Component 2: Alert Level Badge
# Color-coded status display
# ─────────────────────────────────────────────

def classify_glucose(glucose: float) -> dict:
    """Classify glucose level and return display properties."""
    if glucose < 40:
        return {"level": "EMERGENCY", "color": "#CC0000", "emoji": "🚨", "bg": "#FF000022"}
    elif glucose < 54:
        return {"level": "CRITICAL LOW", "color": "#FF0000", "emoji": "🔴", "bg": "#FF000015"}
    elif glucose < 70:
        return {"level": "LOW", "color": "#FF6600", "emoji": "🟠", "bg": "#FF660015"}
    elif glucose <= 180:
        return {"level": "IN RANGE", "color": "#00AA00", "emoji": "🟢", "bg": "#00AA0015"}
    elif glucose <= 250:
        return {"level": "HIGH", "color": "#FF6600", "emoji": "🟠", "bg": "#FF660015"}
    elif glucose <= 350:
        return {"level": "CRITICAL HIGH", "color": "#FF0000", "emoji": "🔴", "bg": "#FF000015"}
    else:
        return {"level": "EMERGENCY", "color": "#CC0000", "emoji": "🚨", "bg": "#FF000022"}


# Demo: show badges for different glucose levels
print("Alert Level Badges:")
print()
for glucose in [38, 48, 65, 120, 200, 280, 380]:
    info = classify_glucose(glucose)
    bar = "█" * int(glucose / 20)
    print(f"  {info['emoji']} {glucose:3d} mg/dL  [{info['level']:14s}]  {bar}")

In [ ]:
# ─────────────────────────────────────────────
# Component 3: Agent Activity Feed
# Shows what each GAS agent is doing
# ─────────────────────────────────────────────

from dataclasses import dataclass
from typing import List

@dataclass
class AgentEvent:
    """A single event in the agent activity feed."""
    timestamp: str
    agent: str
    message: str
    severity: str = "info"  # info, warning, critical


class AgentFeed:
    """Manages the agent activity feed."""

    AGENT_EMOJIS = {
        "Monitor": "👁️",
        "Advisor": "💡",
        "Safety": "🛡️",
        "Caregiver": "📱",
        "Supervisor": "🎯",
    }

    def __init__(self, max_events: int = 10):
        self.events: List[AgentEvent] = []
        self.max_events = max_events

    def add(self, agent: str, message: str, severity: str = "info"):
        """Add a new event to the feed."""
        event = AgentEvent(
            timestamp=datetime.now().strftime("%H:%M:%S"),
            agent=agent,
            message=message,
            severity=severity,
        )
        self.events.insert(0, event)  # newest first
        self.events = self.events[:self.max_events]

    def display(self):
        """Print the feed to console (notebook-friendly)."""
        print("\n🤖 Agent Activity Feed")
        print("─" * 60)
        for event in self.events:
            emoji = self.AGENT_EMOJIS.get(event.agent, "🤖")
            severity_icon = {"info": "ℹ️", "warning": "⚠️", "critical": "🚨"}.get(event.severity, "")
            print(f"  {event.timestamp}  {emoji} {event.agent:10s}  {severity_icon} {event.message}")


# Simulate a hypoglycemia episode feed
feed = AgentFeed()
feed.add("Monitor", "CGM reading: 38 mg/dL (↓↓ falling)", "critical")
feed.add("Supervisor", "Routing to Advisor (critical glucose)", "critical")
feed.add("Advisor", "Generated: EMERGENCY — administer glucagon", "critical")
feed.add("Safety", "Guardrails check: PASSED (urgent language present)", "info")
feed.add("Caregiver", "SMS sent to emergency contact: +1-555-0100", "critical")
feed.add("Monitor", "CGM reading: 42 mg/dL (→ stable)", "warning")
feed.add("Advisor", "Updated: Continue 15g carbs, recheck in 10 min", "warning")
feed.display()

In [ ]:
# ─────────────────────────────────────────────
# Component 4: Human Approval Gate
# HITL pattern for high-stakes decisions
# ─────────────────────────────────────────────

from enum import Enum

class ApprovalStatus(Enum):
    PENDING = "pending"
    APPROVED = "approved"
    REJECTED = "rejected"
    AUTO_APPROVED = "auto_approved"  # for normal-range readings


@dataclass
class ApprovalRequest:
    """A request for human approval of an agent recommendation."""
    id: str
    glucose: float
    recommendation: str
    alert_level: str
    created_at: str
    status: ApprovalStatus = ApprovalStatus.PENDING
    decided_at: str = ""
    decided_by: str = ""


class ApprovalGate:
    """
    Human-in-the-loop approval gate.
    
    For normal glucose: auto-approve.
    For high/critical: require human decision.
    """

    def __init__(self):
        self.requests = []
        self._counter = 0

    def submit(self, glucose: float, recommendation: str, alert_level: str) -> ApprovalRequest:
        """Submit a recommendation for approval."""
        self._counter += 1
        req = ApprovalRequest(
            id=f"req-{self._counter:04d}",
            glucose=glucose,
            recommendation=recommendation,
            alert_level=alert_level,
            created_at=datetime.now().isoformat(),
        )

        # Auto-approve normal readings
        if alert_level == "normal":
            req.status = ApprovalStatus.AUTO_APPROVED
            req.decided_at = req.created_at
            req.decided_by = "system"

        self.requests.append(req)
        return req

    def decide(self, request_id: str, approved: bool, decided_by: str = "human") -> ApprovalRequest:
        """Record a human decision on a pending request."""
        for req in self.requests:
            if req.id == request_id:
                req.status = ApprovalStatus.APPROVED if approved else ApprovalStatus.REJECTED
                req.decided_at = datetime.now().isoformat()
                req.decided_by = decided_by
                return req
        raise ValueError(f"Request {request_id} not found")

    def summary(self):
        """Print approval history summary."""
        print("\n📋 Approval Gate Summary")
        print("─" * 70)
        for req in self.requests:
            status_icon = {
                ApprovalStatus.APPROVED: "✅",
                ApprovalStatus.REJECTED: "❌",
                ApprovalStatus.AUTO_APPROVED: "🔄",
                ApprovalStatus.PENDING: "⏳",
            }[req.status]
            print(f"  {req.id}  {status_icon} {req.status.value:14s}  "
                  f"glucose={req.glucose:3.0f}  [{req.alert_level:8s}]  "
                  f"{req.recommendation[:40]}")


# Demo: simulate approval workflow
gate = ApprovalGate()

# Normal reading: auto-approved
r1 = gate.submit(120, "OK: Glucose within target range.", "normal")

# High reading: requires human
r2 = gate.submit(65, "WARNING: Consume 15g carbs.", "high")
gate.decide(r2.id, approved=True, decided_by="nurse_alice")

# Critical reading: requires human
r3 = gate.submit(42, "EMERGENCY: Administer glucagon.", "critical")
gate.decide(r3.id, approved=True, decided_by="dr_smith")

# Another high reading: rejected (patient already ate)
r4 = gate.submit(68, "CAUTION: Consider 10g carbs.", "high")
gate.decide(r4.id, approved=False, decided_by="patient_alice")

gate.summary()

### Gradio Alternative

For quick prototypes, Gradio offers a 5-line interface:

In [ ]:
# ─────────────────────────────────────────────
# Gradio Alternative (5-line interface)
# ─────────────────────────────────────────────

# NOTE: This code is shown for reference. To run it:
#   pip install gradio
#   Then uncomment and run in a separate script.

GRADIO_CODE = '''
import gradio as gr

def gas_interface(patient_id: str, glucose: float) -> tuple:
    """Run GAS and return recommendation + alert level."""
    # In production: run the full GAS pipeline
    if glucose < 54:
        recommendation = "CRITICAL: Consume 15g fast-acting carbs immediately."
        alert_level = "🔴 CRITICAL"
    elif glucose < 70:
        recommendation = "WARNING: Glucose below target. Consume 10-15g carbs."
        alert_level = "🟠 HIGH"
    elif glucose <= 180:
        recommendation = "OK: Glucose within target range. Continue management."
        alert_level = "🟢 NORMAL"
    else:
        recommendation = "CAUTION: Glucose above target. Check for missed bolus."
        alert_level = "🟠 HIGH"
    return recommendation, alert_level

demo = gr.Interface(
    fn=gas_interface,
    inputs=["text", "number"],
    outputs=["text", "text"],
    title="Guardian Angel System",
    description="Enter patient ID and current glucose to get a recommendation.",
    examples=[["alice-001", 45], ["alice-001", 120], ["bob-002", 280]],
)

# demo.launch()  # Uncomment to run
'''

print("Gradio interface code:")
print(GRADIO_CODE)

# Test the logic without Gradio
def gas_interface(patient_id: str, glucose: float) -> tuple:
    if glucose < 54:
        return "CRITICAL: Consume 15g fast-acting carbs immediately.", "🔴 CRITICAL"
    elif glucose < 70:
        return "WARNING: Glucose below target. Consume 10-15g carbs.", "🟠 HIGH"
    elif glucose <= 180:
        return "OK: Glucose within target range. Continue management.", "🟢 NORMAL"
    else:
        return "CAUTION: Glucose above target. Check for missed bolus.", "🟠 HIGH"

print("\nTesting gas_interface logic:")
for glucose in [45, 65, 120, 220]:
    rec, level = gas_interface("alice-001", glucose)
    print(f"  glucose={glucose:3d}: {level} — {rec[:50]}")

### Running the Dashboard

The complete Streamlit app is saved at `ch28_agentic_ui/app.py`.

**Prerequisites:**
```bash
pip install streamlit plotly
```

**Run:**
```bash
cd /path/to/agentic-ai-book/part5_agentic/ch28_agentic_ui
streamlit run app.py
```

**What you'll see:**
1. A sidebar with patient selector (Alice T1D, Bob T2D)
2. A 24h glucose chart with target range shaded green
3. A real-time status card with current glucose and trend
4. An agent activity feed showing what each GAS agent is doing
5. A recommendation panel with APPROVE/REJECT buttons for high/critical alerts

**Streamlit vs Gradio — When to Use Each:**

| Use Case | Recommendation |
|----------|---------------|
| Patient-facing dashboard | ✅ Streamlit |
| Quick model demo | ✅ Gradio |
| Complex multi-panel layout | ✅ Streamlit |
| API endpoint with UI | ✅ Gradio |
| Production medical app | ✅ Streamlit (or custom React) |
| Research prototype | ✅ Gradio |

## 🏥 GAS Connection

### The Streamlit Dashboard IS the GAS Patient-Facing Interface

The `app.py` you just reviewed is not a demo — it's the actual GAS patient dashboard:

```
GAS Architecture
├── Backend (Python)
│   ├── Monitor Agent    → reads CGM every 5 min
│   ├── Advisor Agent    → generates recommendations
│   ├── Safety Agent     → applies guardrails
│   └── Caregiver Agent  → sends alerts
│
└── Frontend (Streamlit)
    ├── Glucose Chart    → shows what Monitor sees
    ├── Status Card      → shows current alert level
    ├── Agent Feed       → shows what all agents are doing
    └── Approval Gate    → human-in-the-loop for high alerts
```

### Why the UI Matters for Patient Safety

1. **Transparency builds trust** — patients and caregivers can see *why* the agent made a decision
2. **Approval gates prevent harm** — humans must approve before high-stakes actions execute
3. **Override controls are essential** — patients know their bodies; the agent doesn't always
4. **Activity feed reduces anxiety** — knowing the agent is watching 24/7 is reassuring

### The HITL Design Pattern

```
Agent generates recommendation
         ↓
Is alert level HIGH or CRITICAL?
    YES → Show approval gate → Human decides → Log decision
    NO  → Auto-approve → Execute immediately
```

This pattern ensures that **the agent handles routine cases automatically** while **humans retain control over high-stakes decisions**.

## ✅ Checkpoint

### What You Built

| Component | Purpose | Key Pattern |
|-----------|---------|-------------|
| Plotly glucose chart | Visualize 24h CGM data | Target range shading |
| Alert level badge | Color-coded status | Severity → color mapping |
| Agent activity feed | Show agent reasoning | Transparency pattern |
| Approval gate | Human-in-the-loop | HITL pattern |
| Gradio alternative | Quick prototype | 5-line interface |
| `app.py` | Complete Streamlit app | Production-ready |

### Key Takeaways

1. **Agentic UIs are proactive** — they push information to users, not just respond to clicks
2. **Transparency builds trust** — show the agent's reasoning, not just its outputs
3. **Approval gates are safety features** — design them in from the start for high-stakes domains
4. **Streamlit for dashboards, Gradio for demos** — choose the right tool for the job
5. **Session state is your friend** — use `st.session_state` to persist approval history

### Self-Check Questions

1. What happens if the human rejects a recommendation? How should the agent respond?
2. How would you add a "confidence score" to the recommendation display?
3. What's the difference between an approval gate and an override control?
4. How would you make the dashboard update automatically every 5 minutes?

### Congratulations!

You've completed Part 5 of the Agentic AI book. You now know how to:
- Build production agent harnesses (Ch. 18)
- Engineer reliable loops (Ch. 19)
- Model workflows as graphs (Ch. 20)
- Apply design patterns (Ch. 21)
- Manage agent environments (Ch. 22)
- Use MCP for tool integration (Ch. 23)
- Build reusable skill libraries (Ch. 24)
- Enable agent-to-agent communication (Ch. 25)
- Orchestrate multi-agent systems (Ch. 26)
- Choose the right framework (Ch. 27)
- **Build patient-facing agentic UIs (Ch. 28)** ✅